# 🔧 Aula 07 — Múltiplas Ferramentas e MCP
## Guilda de IA — Introdução à IA Generativa

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/aula07_ferramentas_multiplas_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Objetivos:**
1. Chamadas encadeadas com `create_agent`
2. Descrições conflitantes e debug de ferramentas
3. Ferramentas externas via MCP
4. Async em prática

## 🏗️ Setup

⚠️ Vá em `Runtime → Change runtime type` → **T4 GPU**

Execute a célula abaixo e aguarde — leva ~2 min na primeira vez.

In [ ]:
# ── Setup: Ollama + gemma4:e2b-it-qat + deps ──────────────────
# Tudo em uma célula. Execute e prossiga.

# 1. Instalar Ollama + deps Python
!apt-get install -y zstd pciutils lshw > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q langchain langchain-openai langchain-core langgraph langchain-mcp-adapters mcp nest_asyncio requests

# 2. Workaround GPU Colab + keep alive
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

# 3. Iniciar servidor
!pkill -f ollama 2> /dev/null; sleep 1
import subprocess
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env={**os.environ})

# 4. Aguardar servidor
import time, requests
for i in range(30):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).status_code == 200:
            break
    except:
        time.sleep(1)

# 5. Baixar modelo (gemma4:e2b-it-qat = Gemma 4 E2B com QAT, suporta tool calling)
!ollama pull gemma4:e2b-it-qat

# 6. Warm up
print("🔥 Warm up...")
start = time.time()
!curl -s http://localhost:11434/api/chat -d '{"model":"gemma4:e2b-it-qat","messages":[{"role":"user","content":"Hi"}],"stream":false,"keep_alive":-1}' > /dev/null
print(f"✅ Pronto em {time.time()-start:.1f}s")

## 1. Ferramentas (review da S06)

Relembrando: `@tool` + Pydantic = schema automático + validação.

⚠️ Nomes e descrições em **inglês** — modelos entendem tool calling melhor em inglês.

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

@tool
def calculate(a: float, b: float, operation: str) -> float:
    """Performs a mathematical operation between two numbers.
    Use when you need to perform numeric calculations.
    Args:
        a: First number
        b: Second number
        operation: One of 'add', 'subtract', 'multiply', 'divide'
    """
    ops = {'add': a+b, 'subtract': a-b, 'multiply': a*b, 'divide': a/b}
    return ops[operation]

@tool
def get_time() -> str:
    """Returns the current date and time.
    Use when the user asks about the current date, time, or 'what time is it'.
    """
    from datetime import datetime
    return datetime.now().strftime("%d/%m/%Y %H:%M:%S")

@tool
def get_word_length(word: str) -> int:
    """Returns the number of characters in a word.
    Use when the user asks about the length of a word or how many letters it has.
    """
    return len(word)

llm = ChatOpenAI(
    model="gemma4:e2b-it-qat",
    base_url="http://localhost:11434/v1",
    api_key="nao_precisa",
    temperature=0,
)

print("✅ Ferramentas definidas e LLM conectado")

## 2. Chamadas encadeadas

`create_agent` repete Thought→Action→Observation automaticamente quando a pergunta requer 2+ ferramentas em sequência.

In [ ]:
from langchain.agents import create_agent

ferramentas = [calculate, get_time, get_word_length]
agente = create_agent(llm, ferramentas)

# Pergunta que requer encadeamento: multiplicar + somar
resultado = agente.invoke({"messages": "Quanto é 234 vezes 987 mais 100?"})
print(resultado["messages"][-1].content)

## 3. Descrições conflitantes

Duas ferramentas com descrições vagas → LLM confunde.
Duas ferramentas com descrições específicas → LLM distingue.

In [ ]:
# ❌ Descrições vagas
@tool
def find_word_meaning_bad(query: str) -> str:
    """Search for information about a word."""
    return f"Definition of {query}: ..."

@tool
def find_word_rhymes_bad(query: str) -> str:
    """Search for information about a word."""
    return f"Rhymes of {query}: ..."

# ✅ Descrições específicas
@tool
def find_word_meaning(query: str) -> str:
    """Look up the definition and meaning of a word in a dictionary.
    Use when the user asks 'what does X mean' or 'definition of X'.
    """
    return f"Definition of {query}: a word meaning..."

@tool
def find_word_rhymes(query: str) -> str:
    """Find words that rhyme with a given word.
    Use when the user asks for rhymes, similar-sounding words, or poetry help.
    """
    return f"Rhymes of {query}: ..."

# Comparar
agente_vago = create_agent(llm, [find_word_meaning_bad, find_word_rhymes_bad])
r1 = agente_vago.invoke({"messages": "What does 'serendipity' mean?"})
print("Com descrições vagas:")
print(r1["messages"][-1].content)

agente_especifico = create_agent(llm, [find_word_meaning, find_word_rhymes])
r2 = agente_especifico.invoke({"messages": "What does 'serendipity' mean?"})
print("\nCom descrições específicas:")
print(r2["messages"][-1].content)

## 4. Debug de ferramentas

Problemas comuns:
- LLM chama ferramenta errada → melhore a descrição
- Argumentos errados → documente valores possíveis
- ValidationError → Pydantic mostra erro claro (bom sinal)

In [ ]:
perguntas_debug = [
    "Que horas são?",
    "Quanto é 100 + 200?",
    "Quantas letras tem 'Python'?",
]

agente_debug = create_agent(llm, [calculate, get_time, get_word_length])

for pergunta in perguntas_debug:
    print(f"\n📝 {pergunta}")
    resultado = agente_debug.invoke({"messages": pergunta})
    print(f"   → {resultado['messages'][-1].content}")

## 5. Ferramentas externas via MCP

**MCP** (Model Context Protocol) = protocolo padrão para ferramentas de IA.

Passo 1: criar servidor MCP com FastMCP.
Passo 2: consumir com `langchain-mcp-adapters`.

In [ ]:
# ── Servidor MCP ──────────────────────────────────────────────────
server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("math_server.py", "w") as f:
    f.write(server_code)

print("✅ Servidor MCP salvo em math_server.py")

Passo 2: consumir o servidor MCP.

⚠️ MCP é async. No Colab, `await` funciona direto.
Rode `import nest_asyncio; nest_asyncio.apply()` se der erro de event loop.

⚠️ **Bug do Colab:** O kernel Jupyter não suporta `fileno()` em stderr, causando `UnsupportedOperation`. A célula abaixo inclui o workaround.

In [ ]:
import subprocess
import nest_asyncio
nest_asyncio.apply()

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

# ⚠️ No Colab, stderr do kernel não tem fileno().
# Passamos errlog=subprocess.DEVNULL para evitar UnsupportedOperation.
server_params = StdioServerParameters(
    command="python",
    args=["math_server.py"],
)

async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        mcp_tools = await load_mcp_tools(session)

        # Combinar: @tool + MCP
        todas = [calculate, get_time] + mcp_tools
        agente_mcp = create_agent(llm, todas)

        resultado = await agente_mcp.ainvoke(
            {"messages": "Quanto é 3 + 5 multiplicado por 2?"}
        )
        print(resultado["messages"][-1].content)

## 6. MultiServerMCPClient

Conectando dois servidores MCP ao mesmo agente.

In [ ]:
# ── Segundo servidor MCP (strings) ──────────────────────────────
string_server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Strings")

@mcp.tool()
def reverse_string(text: str) -> str:
    """Reverse a string. Use when the user wants to flip or reverse text."""
    return text[::-1]

@mcp.tool()
def count_words(text: str) -> int:
    """Count the number of words in a text.
    Use when the user asks how many words are in a sentence.
    """
    return len(text.split())

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("string_server.py", "w") as f:
    f.write(string_server_code)

print("✅ Servidor de strings salvo em string_server.py")

In [ ]:
# ⚠️ No Colab, sys.stderr não tem fileno(), causando UnsupportedOperation.
# Workaround: redirecionar stderr do processo MCP para /dev/null.
import subprocess
import sys
import os

# Monkey-patch para Colab: garantir que sys.stderr tenha fileno()
if not hasattr(sys.stderr, 'fileno') or not callable(getattr(sys.stderr, 'fileno', None)):
    sys.stderr = open(os.devnull, 'w')

from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "math": {
        "command": "python",
        "args": ["math_server.py"],
        "transport": "stdio",
    },
    "strings": {
        "command": "python",
        "args": ["string_server.py"],
        "transport": "stdio",
    }
})

mcp_tools_multi = await client.get_tools()
print(f"Ferramentas MCP: {[t.name for t in mcp_tools_multi]}")

todas_as_ferramentas = [calculate, get_time] + mcp_tools_multi
agente_multi = create_agent(llm, todas_as_ferramentas)

resultado = await agente_multi.ainvoke(
    {"messages": "Quanto é 10 + 5? E inverta a palavra 'Guilda'."}
)
print(resultado["messages"][-1].content)

---

## 📝 Resumo

- **Chamadas encadeadas**: `create_agent` repete Thought→Action→Observation automaticamente
- **Descrições específicas**: cada ferramenta precisa de descrição que explique *quando* usar
- **Debug**: descrições vagas → ferramenta errada; args mal documentados → valores errados
- **MCP servidor**: `FastMCP` + `@mcp.tool()` + `transport="stdio"`
- **MCP cliente**: `stdio_client` + `ClientSession` + `load_mcp_tools(session)`
- **Multi-server**: `MultiServerMCPClient` combina vários servidores
- **Async**: `ainvoke()` + `await` direto no Colab

**Próxima aula:** RAG — buscar informações em documentos.